# Spatiotemporal feature-engineering shotgun — XGBoost

This notebook screens **all registered feature families** against your existing selected-binary XGBoost baseline. The baseline already contains historical comparable-sale features. Every feature variant is fitted on the same training sample and predicted on the same unseen held-out homes.

**Default experiment:** all `SCREEN_FEATURE_VARIANTS` (including the six new advanced families), 10,000 training sales, up to 1,000 August 2025 validation sales, and 220 XGBoost trees. Residual corrections are **disabled by default**, since we already evaluated those separately. Switch `RUN_RESIDUALS=True` in the settings cell only if you want to rerun the older residual family experiments.

Results are exploratory: these sampled, reduced-tree scores should **not** match the full-history 700-tree XGBoost (7.82% MdAPE). The validation month must be before December 2025; reserve later months for confirmation, since December 2025–May 2026 has already been used repeatedly.

**Files:** `summary.csv`, `predictions.csv`, `failures.json` in the unique `OUTPUT_DIR` below. The experiment does not modify saved baseline models or the production prediction pipeline.


In [1]:
import pandas as pd
from IPython.display import display
from spacetime_fast import run_shotgun, SCREEN_FEATURE_VARIANTS
from models.spacetime_screen_residuals import RESIDUAL_VARIANTS

VALIDATION_MONTH = "2025-08"
TRAIN_START = "2022-01-01"
TRAIN_CAP = 10000             # sample of historical completed sales
VALIDATION_CAP = 1000         # holdout sample, excluded from all training
SCREEN_TREES = 220           # small screen; full model uses 700
SEED = 42
OUTPUT_DIR = "outputs/spacetime_features_all"

RUN_RESIDUALS = False        # residual corrections tested separately
CALIBRATION_CAP = 650        # used only when RUN_RESIDUALS is True
CALIBRATION_FOLDS = 2
RESIDUAL_STRENGTH = 0.5

FEATURES = list(SCREEN_FEATURE_VARIANTS)
RESIDUALS = list(RESIDUAL_VARIANTS) if RUN_RESIDUALS else []

print(f"Feature candidates ({len(FEATURES)}):", FEATURES)
print("Residual candidates:", RESIDUALS)
print("Output:", OUTPUT_DIR)


Feature candidates (16): ['baseline', 'market', 'multiscale', 'time_adjusted_comp', 'spacetime_rbf', 'spacetime_splines', 'combined', 'adaptive_comps', 'hedonic_index', 'geo_context', 'property_spacetime', 'kalman_market', 'advanced_combined', 'comp_age_first', 'comp_type_quota', 'comp_both']
Residual candidates: []
Output: outputs/spacetime_features_all


In [2]:
scores, predictions, failures = run_shotgun(
    validation_month=VALIDATION_MONTH,
    train_start=TRAIN_START,
    train_cap=TRAIN_CAP,
    validation_cap=VALIDATION_CAP,
    calibration_cap=CALIBRATION_CAP,
    calibration_folds=CALIBRATION_FOLDS,
    screen_trees=SCREEN_TREES,
    seed=SEED,
    features=FEATURES,
    residuals=RESIDUALS,
    residual_strength=RESIDUAL_STRENGTH,
    output_dir=OUTPUT_DIR,
)
display(scores)
if failures:
    print(f"{len(failures)} failed candidates; see {OUTPUT_DIR}/failures.json")
    display(pd.DataFrame(failures)[["arm", "variant", "error"]])
else:
    print("All requested candidates finished successfully.")


Shotgun: 10,000 historical sales; 1,000 unseen 2025-08 homes; 220 trees per XGBoost
  feature  baseline              MdAPE 10.769% MAPE 13.949% (ΔMdAPE +0.000pp)
  feature  market                MdAPE 10.851% MAPE 14.058% (ΔMdAPE +0.082pp)


KeyboardInterrupt: 

In [ ]:
# Compare feature candidates to the same reduced-sample XGBoost baseline.
display(scores.sort_values(["arm", "mdape"]))
display(scores.groupby("arm")[["mdape", "mape", "delta_mdape_pp", "delta_mape_pp"]]
        .agg(["min", "median"]).round(3))

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, max(5, 0.4 * len(scores) + 1)))
p = scores.sort_values("delta_mdape_pp")
ax.barh(p["arm"] + " / " + p["variant"], p["delta_mdape_pp"])
ax.axvline(0, linewidth=1, color="gray")
ax.set_xlabel("MdAPE change from baseline (percentage points; lower is better)")
ax.set_title("Spatiotemporal feature shotgun")
fig.tight_layout()
plt.show()


In [ ]:
# Inspect whether variants help particular property types.
import data
sales = data.load("listings")
sold_month = pd.to_datetime(sales["date_sold"]).dt.to_period("M")
names = sales.loc[sold_month == pd.Period(VALIDATION_MONTH),
                  ["ml_num", "house_type_name"]]
annotated = predictions.merge(names, on="ml_num", how="left", validate="many_to_one")
subgroups = (annotated.groupby(["arm", "variant", "house_type_name"])
             .agg(n=("ml_num", "size"), mdape=("abs_pct_error", "median"))
             .query("n >= 10")
             .sort_values(["house_type_name", "mdape"]))
display(subgroups.head(50))
print(f"Full result CSV: {OUTPUT_DIR}/predictions.csv")


## Next: refine the winners

Review `summary.csv` and `failures.json`. Keep 3–5 candidates with the strongest MdAPE/MAPE and reasonable runtime. Confirm them across multiple **earlier** validation months in `spacetime_three_month.ipynb` using 40,000 training examples and 500 trees; for that notebook set `FEATURE_VARIANTS` to the shortlist and `RESIDUAL_VARIANTS=[]`.

A small change on one month can be noise or selection bias. Only promote persistent improvements to full-history 700-tree validation. The baseline and shortlisted variants must use identical chronological splits and target definitions.

If your local repo is behind, update it and open this notebook in Jupyter:

```powershell
git switch main
git pull origin main
uv sync
uv run jupyter lab
```

For a quicker initial screen, change `TRAIN_CAP=5000`, `VALIDATION_CAP=400`, and `SCREEN_TREES=100`; for a stronger screen increase those settings, but retain the same holdout across variants.


---

## Structural spatiotemporal shotgun — GPBoost, Matérn GP, local kernels

**This is a separate experiment from the feature-engineering shotgun above.** Run the new cells below; the earlier feature-screening cells do **not** need to be rerun.

We test actual model structures, not just new features:

| Model | Role |
| --- | --- |
| `baseline` | Matched tuned **XGBoost + splines + geo** |
| `xgb_tensor_gam` | Matched XGBoost + splines + geo **+ Tensor GAM** (strong incumbent) |
| `xgb_st_kernel` / `xgb_st_kernel_type` | Out-of-time local space-time kernel corrections |
| `xgb_spatial_matern` | Exact spatial Matérn GP correction |
| `xgb_st_matern` | Exact three-dimensional space-time Matérn GP correction |
| `gpboost_vecchia` | Independent joint boosted-trees + spatiotemporal GP model (optional package) |

All methods use the **same sampled training period** and **identical held-out homes**, with strict chronological cutoffs. Corrections use *earlier out-of-time errors*, never their own training errors.

**Screening is not final validation.** Our full-history off-market best remains **7.4857% MdAPE**. The pilot trains fewer XGBoost trees and sales; compare *deltas against this pilot's matched incumbent*, **not** raw MdAPE against the full-history best. A one-month smoke test is for functional checks only. The three-month test helps select candidates for full-history confirmation.

Implementations: `models/spacetime_structural_features.py`, `models/spacetime_structural_residuals.py`, `models/spacetime_gpboost.py`. The root `spacetime_structural_shotgun.py` only coordinates evaluation, checkpointing and comparisons. GPBoost requires `%pip install gpboost` once; if absent it is logged as unavailable rather than counted as an inferior model.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from spacetime_structural_shotgun import run_structural_shotgun, METHODS
from models.spacetime_structural_features import SPLINE_GEO_SUFFIX

assert SPLINE_GEO_SUFFIX == "__st_spacetime_splines+geo_context"
assert "gpboost_vecchia" in METHODS
assert "xgb_tensor_gam" in METHODS

# Independent output folders prevent the small smoke run from contaminating
# a larger three-month experiment. Each run resumes only identical settings.
RUN_STRUCTURAL_SMOKE = True
SMOKE_SETTINGS = dict(
    months=("2025-10",),
    train_start="2022-01-01",
    train_cap=5_000,
    validation_cap=250,
    calibration_cap=150,
    calibration_folds=2,
    trees=250,
    gpboost_rounds=180,
    max_gp_points=250,
    residual_strength=0.5,
    methods=METHODS,
    seed=42,
    bootstrap_repeats=250,
    output_dir="outputs/spacetime_structural_notebook_smoke",
    resume=True,
)
print("Structural candidates:", ", ".join(METHODS))
print("Pilot data/trees:", SMOKE_SETTINGS["train_cap"], SMOKE_SETTINGS["trees"])


In [ ]:
# STEP 1 — one-month smoke / initial comparison.
# Run this cell before the full three-month pilot.
if RUN_STRUCTURAL_SMOKE:
    structural_smoke, structural_smoke_monthly, structural_smoke_predictions, structural_smoke_failures = (
        run_structural_shotgun(**SMOKE_SETTINGS)
    )
    display(structural_smoke[[
        "variant", "homes", "mdape", "mape",
        "delta_mdape_pp", "delta_mape_pp",
        "runtime_seconds", "triage",
    ]].round(4))

    if not structural_smoke_failures.empty:
        print("Unavailable/failed variants (NOT classified as model losses):")
        display(structural_smoke_failures[[
            "month", "variant", "error_type", "message"
        ]])
    print("Smoke outputs:", SMOKE_SETTINGS["output_dir"])


In [ ]:
# STEP 2 — inspect which models are even plausible against our incumbent.
# Negative change means an improvement. This cell uses the smoke-run results.
if RUN_STRUCTURAL_SMOKE and "structural_smoke" in globals():
    columns = [
        "variant", "delta_mdape_pp", "delta_mape_pp",
        "delta_vs_tensor_mdape_pp", "delta_vs_tensor_mape_pp",
        "runtime_seconds", "triage",
    ]
    display(structural_smoke[[
        c for c in columns if c in structural_smoke.columns
    ]].round(4))

    if "delta_vs_tensor_mdape_pp" in structural_smoke:
        plot_data = structural_smoke.sort_values("delta_vs_tensor_mdape_pp")
        fig, ax = plt.subplots(figsize=(9, max(4, 0.55 * len(plot_data))))
        ax.barh(plot_data.variant, plot_data.delta_vs_tensor_mdape_pp)
        ax.axvline(0, color="gray", linewidth=1)
        ax.invert_yaxis()
        ax.set_xlabel("ΔMdAPE vs matched spline + geo + Tensor GAM (pp; lower better)")
        ax.set_title("Structural space-time smoke test — exploratory")
        fig.tight_layout()
        plt.show()

    print("One month cannot reliably eliminate model classes.")
    print("Use the three-month pilot below before shortlisting models.")


### Three-month development pilot (opt-in)

After the smoke test succeeds, change `RUN_STRUCTURAL_THREE_MONTH` to `True` and run the cell below. This uses 8,000 past sales, 500 trees and held-out sales from September, October and November 2025. It will take more computation than the smoke test.

Prioritize methods that repeatedly improve **both MdAPE and MAPE**, have acceptable runtime, and beat the **matched Tensor GAM** rather than just the uncorrected XGBoost. The `triage` column is a conservative screening heuristic, not proof that any model can or cannot win after tuning.

Changing training budgets changes the experimental fingerprint: use a different `output_dir` instead of mixing caches. Completed monthly predictions are checkpointed and reused with `resume=True`.


In [ ]:
# STEP 3 — more discriminating screen on three earlier monthly holdouts.
RUN_STRUCTURAL_THREE_MONTH = False   # Enable after reviewing smoke output

THREE_MONTH_SETTINGS = dict(
    months=("2025-09", "2025-10", "2025-11"),
    train_start="2022-01-01",
    train_cap=8_000,
    validation_cap=900,
    calibration_cap=450,
    calibration_folds=2,
    trees=500,
    gpboost_rounds=220,
    max_gp_points=350,
    residual_strength=0.5,
    methods=METHODS,
    seed=42,
    bootstrap_repeats=500,
    output_dir="outputs/spacetime_structural_notebook_three_month",
    resume=True,
)

if RUN_STRUCTURAL_THREE_MONTH:
    structural_pooled, structural_monthly, structural_predictions, structural_failures = (
        run_structural_shotgun(**THREE_MONTH_SETTINGS)
    )
    display(structural_pooled.round(4))
    display(structural_monthly.round(4))
    if not structural_failures.empty:
        display(structural_failures[[
            "month", "variant", "error_type", "message"
        ]])
    print("Full pilot outputs:", THREE_MONTH_SETTINGS["output_dir"])
